In [2]:
from cobolt.utils import SingleData, MultiomicDataset
from cobolt.model import Cobolt
import os
import gc
import numpy as np
import pandas as pd
import scipy.sparse as sps
import scanpy as sc
import h5py
import scipy.io as sio
import warnings
# from muon import atac as ac
import seaborn as sns

from os.path import join
import matplotlib.pyplot as plt

/home/yanxh/anaconda3/envs/multi-omics/lib/python3.8/site-packages/tqdm-4.65.0-py3.8.egg/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yanxh/anaconda3/envs/multi-omics/lib/python3.8/site-packages/umap_learn-0.5.3-py3.8.egg/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/yanxh/anaconda3/envs/multi-omics/lib/python3.8/site-packages/umap_learn-0.5.3-py3.8.egg/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' 

In [3]:
def set_col2cat(ad, cols=[]):
    for col in cols:
        ad.obs[col] = ad.obs[col].astype('category')

def unify_colors(queries, color_key, ref_color_dict):
    for q in queries:
        q.obs[color_key] = q.obs[color_key].astype('category')
        q.uns[f'{color_key}_colors'] = [ref_color_dict[_] for _ in q.obs[color_key].cat.categories]
    return queries

def get_umap(ad, use_reps=[]):
    for use_rep in use_reps:
        umap_add_key = f'{use_rep}_umap'
        sc.pp.neighbors(ad, use_rep=use_rep, n_neighbors=15)
        sc.tl.umap(ad)
        ad.obsm[umap_add_key] = ad.obsm['X_umap']
    return ad

def wrap_warn_plot(adata, basis, color, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        sc.pl.embedding(adata, basis=basis, color=color, **kwargs)

def wrap_warn_comp_plot(ads, basis, colors, figw=5, figh=4, **kwargs):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for i, color in enumerate(colors):
            _, axes = plt.subplots(1, len(ads), figsize=(figw*len(ads), figh))
            for j, ad in enumerate(ads):
                sc.pl.embedding(ad, basis=basis, color=color, ax=axes[j], show=False, **kwargs)
            plt.show()

In [4]:
def split_ob(ads, ad_ref, ob='obs', key='emb2'):
    len_ads = [_.n_obs for _ in ads]
    if ob=='obsm':
        split_obsms = np.split(ad_ref.obsm[key], np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obsms):
            ad.obsm[key] = v
    else:
        split_obs = np.split(ad_ref.obs[key].to_list(), np.cumsum(len_ads[:-1]))
        for ad, v in zip(ads, split_obs):
            ad.obs[key] = v
    
def subset_ad(ad, subset_index):
    ad = ad[subset_index].copy()
    return ad

def set_spatial(ad):
    ad.obsm['spatial'] = ad.obs[['array_row', 'array_col']].values
    ad.obsm['spatial'] = ad.obsm['spatial'][:, ::-1]
    ad.obsm['spatial'][:, 1] = -1 * ad.obsm['spatial'][:, 1]
    return ad

def reorder(ad1, ad2):
    shared_barcodes = ad1.obs_names.intersection(ad2.obs_names)
    ad1 = ad1[shared_barcodes].copy()
    ad2 = ad2[shared_barcodes].copy()
    return ad1, ad2

def load_peak_expr(_dir):
    data = sio.mmread(join(_dir, 'data.mtx'))
    cname = pd.read_csv(join(_dir, 'barcode.csv'), index_col=0)['x'].to_list()
    feat = pd.read_csv(join(_dir, 'feat.csv'), index_col=0)['x'].to_list()
    ad = sc.AnnData(sps.csr_matrix(data.T))
    ad.obs_names = cname
    ad.var_names = feat
    return ad

In [5]:
data_dir = '../../../data/processed/Mux-VisiumHD-HE'

out_dir = './outputs/Mux-VisiumHD-HE'
os.makedirs(out_dir, exist_ok=True)

In [6]:
ad_mux_rna = sc.read_h5ad(join(data_dir, 'Mux/ad_rna.h5ad'))
ad_mux_atac = sc.read_h5ad(join(data_dir, 'Mux/ad_atac.h5ad'))

ad_vhd_rna = sc.read_h5ad(join(data_dir, 'VisiumHD/ad_rna.h5ad'))
ad_vhe_rna = sc.read_h5ad(join(data_dir, 'VisiumHE/ad_rna.h5ad')) 

In [7]:
for ad in [ad_mux_rna, ad_vhd_rna, ad_vhe_rna]:
    print(ad.shape)

(9686, 18445)
(98917, 18445)
(2702, 18445)


In [9]:
ad_rna_all = sc.concat([ad_mux_rna, ad_vhd_rna, ad_vhe_rna])

sc.pp.highly_variable_genes(ad_rna_all, flavor='seurat_v3', n_top_genes=10000, batch_key='Slice')
hvg_names = ad_rna_all.var.query('highly_variable').index.to_numpy()

<frozen importlib._bootstrap>:219: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 216 from C header, got 232 from PyObject


In [10]:
# ac.pp.tfidf(ad_atac_all, scale_factor=1e4)
sc.pp.highly_variable_genes(ad_mux_atac, flavor='seurat_v3', n_top_genes=100000, batch_key='Slice')
hvp_names = ad_mux_atac.var.query('highly_variable').index.to_numpy()

In [11]:
mux_rna_count = sps.csr_matrix(ad_mux_rna[:, hvg_names].X.astype(np.float32))
mux_atac_count = sps.csr_matrix(ad_mux_atac[:, hvp_names].X.astype(np.float32))

vhd_rna_count = sps.csr_matrix(ad_vhd_rna[:, hvg_names].X.astype(np.float32))
vhe_rna_count = sps.csr_matrix(ad_vhe_rna[:, hvg_names].X.astype(np.float32))

mux_barc = ad_mux_rna.obs_names.to_numpy()
vhd_barc = ad_vhd_rna.obs_names.to_numpy()
vhe_barc = ad_vhe_rna.obs_names.to_numpy()

rna_feature = hvg_names
atac_feature = hvp_names

In [12]:
mux_rna = SingleData("GeneExpr", "Mux", rna_feature, mux_rna_count, mux_barc)
mux_atac = SingleData("ATAC", "Mux", atac_feature, mux_atac_count, mux_barc)

vhd_rna = SingleData("GeneExpr", "VHD", rna_feature, vhd_rna_count, vhd_barc)
vhe_rna = SingleData("GeneExpr", "VHE", rna_feature, vhe_rna_count, vhe_barc)

In [13]:
multi_dt = MultiomicDataset.from_singledata(
    mux_rna, mux_atac, vhd_rna, vhe_rna)
print(multi_dt)

A MultiomicDataset object with 2 omics:
- GeneExpr: 10000 features, 111305 cells, 3 batches.
- ATAC: 47125 features, 9686 cells, 1 batches.
Joint cells:
- GeneExpr, ATAC: 9686 cells.


### Training

In [ ]:
model = Cobolt(dataset=multi_dt, lr=0.001, n_latent=10, batch_size=128) 
model.train(num_epochs=100)

In [ ]:
plt.plot(model.history['loss'])

In [ ]:
model.calc_all_latent()

latent = model.get_all_latent()

In [14]:
if False:
    latent_barcode = np.array([_.split('~')[1] for _ in latent[1]])
    df_latent = pd.DataFrame(latent[0], index=latent_barcode)
    df_latent.to_csv(f'{out_dir}/res.csv')

else:
    df_latent = pd.read_csv(f'{out_dir}/res.csv', index_col=0)
    df_latent.index = df_latent.index.astype('str')

In [15]:
ad_mux_rna.obsm['X_emb'] = df_latent.loc[ad_mux_rna.obs_names, :].values
ad_vhd_rna.obsm['X_emb'] = df_latent.loc[ad_vhd_rna.obs_names, :].values
ad_vhe_rna.obsm['X_emb'] = df_latent.loc[ad_vhe_rna.obs_names, :].values

In [16]:
ad_mosaic = sc.concat([ad_mux_rna, ad_vhd_rna, ad_vhe_rna])
# ad_mosaic = get_umap(ad_mosaic, ['X_emb'])

In [20]:
anal_dir = '../../../results/embeddings/Cobolt/Mux-VisiumHD-HE'
os.makedirs(anal_dir, exist_ok=True)

In [21]:
df = pd.DataFrame(ad_mosaic.obsm['X_emb'], index=ad_mosaic.obs_names)
df.to_csv(join(anal_dir, 'df_emb.csv'))